In [2]:
print('test')

test


In [4]:
from langchain.agents import create_agent
from IPython.display import Markdown
from dotenv.ipython import load_dotenv
from langchain_openai import ChatOpenAI

In [6]:
load_dotenv(override=True)

True

In [14]:
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=1000)

In [29]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [30]:
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="You are a helpful assistant",
)

In [31]:
resp=agent.invoke(input={"messages" : [
    {"role":"user", "content":"Je m'appelle Rahma"}
    ]})


In [32]:
print(resp["messages"] [-1].content)

Bonjour Rahma ! Comment puis-je vous aider aujourd'hui ?


In [22]:
response = agent.invoke(
    input={
        'messages':[
              {'role':'user', 'content':'What is My Name'}
             ]
            }
            )

In [23]:
print(display(Markdown(response['messages'][-1].content)))

I'm sorry, but I don't have access to personal data or the ability to know your name. If you tell me your name, I can address you by it in our conversation!

None


In [42]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [25]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=1000)
advenced_llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=1000)

In [38]:
@wrap_model_call
def dynamic_model_selection(request, handler)->ModelResponse:
    env = request.runtime.context.get("env", "test")
    if env == "prod":
        model = advenced_llm
        print("advenced_llm selected")
    else:
        model = basic_llm
        print("basic_llm selected")
    return handler(request.override(model=model))

In [56]:
agent2 = create_agent(
    model=basic_llm,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

In [57]:
resp=agent2.invoke(
    input={"messages":[HumanMessage("What is ai an agent")]},
    context={"env": "prod"}
)


[values] {'messages': [HumanMessage(content='What is ai an agent', additional_kwargs={}, response_metadata={}, id='10834e9c-6957-4901-81ff-b8a6b657807b')]}
advenced_llm selected
[updates] {'model': {'messages': [AIMessage(content='An AI agent is a software entity that performs tasks autonomously using artificial intelligence techniques. It perceives its environment through sensors, processes the information, and takes actions to achieve specific goals. AI agents can range from simple programs, like chatbots, to complex systems, like autonomous vehicles. They are designed to operate with varying degrees of independence and can learn from their experiences to improve performance over time. AI agents are used in various applications, including robotics, virtual assistants, recommendation systems, and more.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 12, 'total_tokens': 111, 'completion_tokens_details': {'accepted_pred

In [58]:
from IPython.display import Markdown, display

In [59]:
print(display(Markdown(resp["messages"] [-1].content)))

An AI agent is a software entity that performs tasks autonomously using artificial intelligence techniques. It perceives its environment through sensors, processes the information, and takes actions to achieve specific goals. AI agents can range from simple programs, like chatbots, to complex systems, like autonomous vehicles. They are designed to operate with varying degrees of independence and can learn from their experiences to improve performance over time. AI agents are used in various applications, including robotics, virtual assistants, recommendation systems, and more.

None


# Agent Memory

In [63]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

In [67]:
agent = create_agent(
  model="openai:gpt-5.2",
  system_prompt="You are a helpful assistant",
)

In [68]:
resp=agent.invoke(
  input={'messages':[HumanMessage('My Name Rahma')]},
  config={'configurable':{'thread_id':1}}
)
print(resp['messages'][-1].content)

Hi Rahma. Nice to meet you—what would you like help with today?


In [69]:
resp=agent.invoke(
  input={'messages':[HumanMessage('What is my name')]},
  config={'configurable':{'thread_id':1}}
)
print(resp['messages'][-1].content)


I don’t know your name from this chat. If you tell me what you’d like to be called, I’ll use it.


In [ ]:
memory = InMemorySaver()
agent = create_agent(
  model="openai:gpt-5.2",
  system_prompt="You are a helpful assistant",
  checkpointer=InMemorySaver()
)